# 02 - Spike Encoding Visualisation

Visualise how different spike encoding methods convert mel-spectrograms into spike trains.
This notebook compares: rate coding, delta coding, latency coding, and direct encoding.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import librosa
import librosa.display

from src.config import ESC50_AUDIO_DIR, ESC50_META_PATH, SAMPLE_RATE, HOP_LENGTH, NUM_STEPS
from src.dataset import download_esc50, wav_to_mel_spectrogram, normalise_spectrogram
from src.encoding import encode_rate, encode_delta, encode_latency, encode_direct

import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)

In [ ]:
download_esc50()
meta = pd.read_csv(ESC50_META_PATH)

# Pick a clear example: dog bark
row = meta[meta['category'] == 'dog'].iloc[0]
filepath = ESC50_AUDIO_DIR / row['filename']
print(f"Using: {row['filename']} ({row['category']})")

mel = wav_to_mel_spectrogram(str(filepath))
mel_norm = normalise_spectrogram(mel)
print(f"Spectrogram shape: {mel_norm.shape}")

# Convert to tensor: (1, 1, n_mels, time) for batch=1, channels=1
tensor = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
print(f"Input tensor shape: {tensor.shape}")

In [ ]:
# Generate spike trains with each encoding
spk_rate = encode_rate(tensor)
spk_delta = encode_delta(tensor)
spk_latency = encode_latency(tensor)
spk_direct = encode_direct(tensor)

print(f"Rate encoding shape:    {spk_rate.shape}")
print(f"Delta encoding shape:   {spk_delta.shape}")
print(f"Latency encoding shape: {spk_latency.shape}")
print(f"Direct encoding shape:  {spk_direct.shape}")

# Spike density (fraction of 1s)
print(f"\nSpike density:")
print(f"  Rate:    {spk_rate.mean():.4f}")
print(f"  Delta:   {spk_delta.mean():.4f}")
print(f"  Latency: {spk_latency.mean():.4f}")
print(f"  Direct:  {spk_direct.mean():.4f} (continuous, not binary)")

In [ ]:
# Visualise: spike raster plots for a small region of the spectrogram
# Show spikes for mel bins 20-35 across all timesteps

mel_start, mel_end = 20, 36  # 16 mel bins
time_col = 50  # pick one time frame column

encodings = {
    'Rate': spk_rate[:, 0, 0, mel_start:mel_end, time_col].numpy(),
    'Delta': spk_delta[:, 0, 0, mel_start:mel_end, time_col].numpy(),
    'Latency': spk_latency[:, 0, 0, mel_start:mel_end, time_col].numpy(),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, spk_data) in zip(axes, encodings.items()):
    # spk_data shape: (num_steps, mel_bins)
    for neuron in range(spk_data.shape[1]):
        spike_times = np.where(spk_data[:, neuron] > 0.5)[0]
        ax.scatter(spike_times, [neuron] * len(spike_times), s=3, c='black')
    
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Mel bin')
    ax.set_title(f'{name} Encoding\n(spikes: {int(spk_data.sum())})')
    ax.set_xlim(-0.5, NUM_STEPS - 0.5)
    ax.set_ylim(-0.5, spk_data.shape[1] - 0.5)

plt.suptitle(f'Spike Raster Plots (mel bins {mel_start}-{mel_end}, time frame {time_col})', y=1.02)
plt.tight_layout()
plt.savefig('../results/encoding_raster_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise: spike density heatmaps (averaged across timesteps)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

all_encodings = {
    'Rate': spk_rate[:, 0, 0].mean(dim=0).numpy(),
    'Delta': spk_delta[:, 0, 0].mean(dim=0).numpy(),
    'Latency': spk_latency[:, 0, 0].mean(dim=0).numpy(),
    'Direct (mean input)': spk_direct[:, 0, 0].mean(dim=0).numpy(),
}

for ax, (name, data) in zip(axes.flat, all_encodings.items()):
    im = ax.imshow(data, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'{name} - Mean Activity')
    ax.set_xlabel('Time frame')
    ax.set_ylabel('Mel bin')
    plt.colorbar(im, ax=ax)

plt.suptitle('Spike Density / Activity Heatmaps (averaged over timesteps)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/encoding_heatmaps.png', dpi=150)
plt.show()

In [ ]:
# Compare spike counts across different sound categories
categories_to_check = ['dog', 'rain', 'clock_tick', 'helicopter', 'crying_baby']

results = []
for cat in categories_to_check:
    row = meta[meta['category'] == cat].iloc[0]
    fp = ESC50_AUDIO_DIR / row['filename']
    m = normalise_spectrogram(wav_to_mel_spectrogram(str(fp)))
    t = torch.tensor(m, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    
    results.append({
        'category': cat,
        'rate_spikes': encode_rate(t).sum().item(),
        'delta_spikes': encode_delta(t).sum().item(),
        'latency_spikes': encode_latency(t).sum().item(),
        'mean_intensity': m.mean(),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))